In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
import re
import sys
import unicodedata

import numpy as np
import pandas as pd
from IPython.display import display
import plotly.express as px
import plotly.graph_objects as go
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import RobustScaler

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src").exists():
    ROOT = ROOT.parent

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

try:
    import yaml
except ImportError:
    yaml = None

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 120)


In [ ]:
@dataclass
class SeriesConfig:
    input_path: str = "data/temp/output_clean.csv"
    ruleset_path: str = "configs/rulesets/loreal_cash_in_v1.yaml"
    date_col: str = "VALUE_DATE"
    amount_col: str = "SIGNED_AMOUNT"
    movement_col: str = "CASH_MOVEMENT_TYPE_SHORTNAME"
    movement_value: str = "TRF+"
    id_col: str = "ID"
    company_text_candidates: tuple[str, ...] = ("LABEL", "DESCRIPTION", "CPTY_NAME", "CPTY_SHORTNAME")
    company_col: str = "company_name"


cfg = SeriesConfig()

asdict(cfg)

In [ ]:
def normalize_text(value) -> str | None:
    if pd.isna(value):
        return None
    text = str(value).strip()
    if not text:
        return None
    text = text.upper()
    text = "".join(
        c for c in unicodedata.normalize("NFD", text)
        if unicodedata.category(c) != "Mn"
    )
    text = re.sub(r"[^A-Z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text or None


def normalize_company(name) -> str | None:
    name = normalize_text(name)
    if not name:
        return None
    legal_forms = [
        r"\bS\s*A\s*U\b", r"\bS\s*A\b", r"\bS\s*L\s*U\b", r"\bS\s*L\b",
        r"\bB\s*V\b", r"\bN\s*V\b", r"\bS\s*A\s*R\s*L\b", r"\bS\s*C\s*A\b",
        r"\bC\s*B\b", r"\bCO\s*LTD\b", r"\bCO\s*LIMITED\b", r"\bLTD\b", r"\bLIMITED\b",
    ]
    for pattern in legal_forms:
        name = re.sub(pattern, " ", name)
    name = re.sub(r"\s+", " ", name).strip()
    if not name or re.fullmatch(r"\d+", name):
        return None
    if re.fullmatch(r"(?:\d\s*)+", name[:9].strip()):
        return None
    if name.startswith("INCOMING"):
        return None
    return name or None


def extract_company(text) -> str | None:
    if pd.isna(text):
        return None
    text = str(text).strip()
    if not text:
        return None
    patterns = [
        r"TRANSFERENCIA DE\s+([^,]+)",
        r"/PT/FT/BO/([^/]+)",
        r"/BO/([^/]+)",
        r"^([^/]+)/",
    ]
    for pattern in patterns:
        match = re.search(pattern, text, flags=re.IGNORECASE)
        if match:
            candidate = match.group(1).strip()
            if candidate and not re.fullmatch(r"\d+", candidate):
                return candidate
    return None


def parse_amount_series(series: pd.Series) -> pd.Series:
    if pd.api.types.is_numeric_dtype(series):
        return pd.to_numeric(series, errors="coerce")
    return pd.to_numeric(
        series.astype("string")
        .str.replace("\u00A0", "", regex=False)
        .str.replace(" ", "", regex=False)
        .str.replace(",", ".", regex=False),
        errors="coerce",
    )


def first_non_empty_text(df: pd.DataFrame, columns: tuple[str, ...]) -> pd.Series:
    out = pd.Series(pd.NA, index=df.index, dtype="string")
    for col in columns:
        if col not in df.columns:
            continue
        candidate = df[col].astype("string").str.strip()
        candidate = candidate.mask(candidate.eq(""), pd.NA)
        out = out.fillna(candidate)
    return out


def sanitize_label(value: str, max_len: int = 80) -> str:
    text = normalize_text(value) or "UNKNOWN"
    text = re.sub(r"[^A-Z0-9]+", "_", text).strip("_")
    return (text or "UNKNOWN")[:max_len]

In [ ]:
def normalized_entropy(values) -> float:
    counts = pd.Series(values).value_counts(dropna=True)
    if len(counts) <= 1:
        return 0.0
    p = counts / counts.sum()
    return float(-(p * np.log(p)).sum() / np.log(len(counts)))


def top_share(values) -> float:
    counts = pd.Series(values).value_counts(dropna=True)
    if counts.sum() == 0:
        return np.nan
    return float(counts.iloc[0] / counts.sum())


def safe_cv2(values) -> float:
    arr = pd.Series(values).dropna().astype(float)
    if len(arr) <= 1:
        return np.nan
    mean = arr.mean()
    if abs(mean) < 1e-12:
        return np.nan
    return float((arr.std(ddof=1) / mean) ** 2)


def gap_stats(dates) -> dict[str, float]:
    dates = pd.Series(pd.to_datetime(dates).dropna().unique()).sort_values()
    if len(dates) <= 1:
        return {
            "median_gap_days": np.nan,
            "mean_gap_days": np.nan,
            "cv_gap_days": np.nan,
            "min_gap_days": np.nan,
            "max_gap_days": np.nan,
        }
    gaps = dates.diff().dropna().dt.days.astype(float)
    mean_gap = gaps.mean()
    cv_gap = gaps.std(ddof=1) / mean_gap if mean_gap and not np.isnan(mean_gap) else np.nan
    return {
        "median_gap_days": float(gaps.median()),
        "mean_gap_days": float(mean_gap),
        "cv_gap_days": float(cv_gap) if not np.isnan(cv_gap) else np.nan,
        "min_gap_days": float(gaps.min()),
        "max_gap_days": float(gaps.max()),
    }


def sbc_label(adi: float, cv2: float) -> str:
    """
    Syntetos-Boylan-Croston-style demand pattern label.

    Use as a diagnostic feature, not as a hard truth:
    daily treasury company series are often zero-heavy, so most companies may look intermittent/lumpy.
    """
    if pd.isna(adi) or pd.isna(cv2):
        return "unknown"
    if adi < 1.32 and cv2 < 0.49:
        return "smooth"
    if adi < 1.32 and cv2 >= 0.49:
        return "erratic"
    if adi >= 1.32 and cv2 < 0.49:
        return "intermittent"
    return "lumpy"


def longest_run(mask) -> int:
    best = current = 0
    for value in np.asarray(mask, dtype=bool):
        if value:
            current += 1
            best = max(best, current)
        else:
            current = 0
    return int(best)


def autocorr_at_lag(values, lag: int) -> float:
    series = pd.Series(values).astype(float)
    if len(series) <= lag:
        return np.nan
    left = series.iloc[:-lag]
    right = series.iloc[lag:]
    if left.std(ddof=0) < 1e-12 or right.std(ddof=0) < 1e-12:
        return np.nan
    return float(left.corr(right))


def share_last_days(daily: pd.Series, days: int) -> float:
    total = daily.sum()
    if abs(total) < 1e-12:
        return 0.0
    cutoff = daily.index.max() - pd.Timedelta(days=days - 1)
    return float(daily.loc[daily.index >= cutoff].sum() / total)


In [ ]:
def load_ruleset_payload(path: str | Path) -> dict:
    if yaml is None:
        return {}
    ruleset_path = Path(path)
    if not ruleset_path.is_absolute():
        ruleset_path = ROOT / ruleset_path
    if not ruleset_path.exists():
        return {}
    with ruleset_path.open("r", encoding="utf-8") as handle:
        return yaml.safe_load(handle) or {}


def apply_ruleset_filters_light(df: pd.DataFrame, ruleset_payload: dict) -> tuple[pd.DataFrame, pd.DataFrame]:
    filters = ruleset_payload.get("filters", {}) if ruleset_payload else {}
    current = df.copy()
    rows = []
    for column, rule in filters.items():
        before = len(current)
        if column not in current.columns:
            rows.append({"rule": column, "status": "missing_column", "before": before, "removed": 0, "after": before})
            continue
        case_sensitive = bool(rule.get("case_sensitive", False))
        normalized = current[column].astype("string")
        if not case_sensitive:
            normalized = normalized.str.lower()
        keep = pd.Series(True, index=current.index)

        include_values = rule.get("include_values") or []
        if include_values:
            values = {str(v if case_sensitive else str(v).lower()) for v in include_values}
            keep &= normalized.isin(values)
        exclude_values = rule.get("exclude_values") or []
        if exclude_values:
            values = {str(v if case_sensitive else str(v).lower()) for v in exclude_values}
            keep &= ~normalized.isin(values)
        exclude_contains = rule.get("exclude_contains") or []
        if exclude_contains:
            values = [str(v if case_sensitive else str(v).lower()) for v in exclude_contains]
            pattern = "|".join(re.escape(v) for v in values)
            keep &= ~normalized.fillna("").str.contains(pattern, regex=True, na=False)

        current = current.loc[keep].copy()
        after = len(current)
        rows.append({"rule": column, "status": "applied", "before": before, "removed": before - after, "after": after})
    return current.reset_index(drop=True), pd.DataFrame(rows)


def canonicalize_company_names(tx: pd.DataFrame, threshold: int = 90) -> pd.DataFrame:
    out = tx.copy()
    original = out["company_name"].copy()
    names = original.dropna().value_counts()
    try:
        from rapidfuzz import fuzz

        def score(a, b):
            return fuzz.token_sort_ratio(a, b)
    except ImportError:
        from difflib import SequenceMatcher

        def score(a, b):
            return 100 * SequenceMatcher(None, a, b).ratio()

    mapping = {}
    used = set()
    ordered_names = names.index.tolist()
    for name in ordered_names:
        if name in used:
            continue
        group = [name]
        used.add(name)
        for other in ordered_names:
            if other in used:
                continue
            if score(name, other) >= threshold:
                group.append(other)
                used.add(other)
        best = max(group, key=lambda n: names.loc[n])
        for item in group:
            mapping[item] = best
    manual_mapping = {
        'COFARES SDAD COOPERATIVA FARMACEUT': 'COFARES SOCIEDAD COOPERATIVA FARMA',
        'SANTANDER FACTORING Y CONFIRMING E F C': 'SANTANDER FACTORING Y CONFIRMING S',
        'AYESA IBERMATICAS A U': 'AYESA IBERMATICA',
        'PAYPAL EUROPE R L ET CIE S C A': 'PAYPAL EUROPE R L ET CIE',
    }
    mapping.update(manual_mapping)
    out["company_name"] = original.map(mapping).fillna(original)
    out["company_found"] = out["company_name"].notna()
    out["company_key"] = out["company_name"].fillna("NO_COMPANY")
    return out

In [ ]:
input_path = ROOT / cfg.input_path
df = pd.read_csv(input_path)
print("Raw input:", df.shape)

df.columns = [str(c).strip() for c in df.columns]
ruleset_payload = load_ruleset_payload(cfg.ruleset_path)
df_rules, rule_waterfall = apply_ruleset_filters_light(df, ruleset_payload)
display(rule_waterfall)
print("After YAML ruleset safety filters:", df_rules.shape)

df_trf_clean = df_rules.loc[df_rules[cfg.movement_col].astype("string").eq(cfg.movement_value)].copy()
print(f"After {cfg.movement_value} filter:", df_trf_clean.shape)

for col in ["CAPTURE_DATE", "ISSUE_DATE", "TRADE_DATE", "VALUE_DATE", "MATCH_DATE"]:
    if col in df_trf_clean.columns:
        df_trf_clean[col] = pd.to_datetime(df_trf_clean[col], errors="coerce").dt.normalize()
for col in ["AMOUNT", "SIGNED_AMOUNT", "ORIGIN_AMOUNT"]:
    if col in df_trf_clean.columns:
        df_trf_clean[col] = parse_amount_series(df_trf_clean[col])

if cfg.id_col in df_trf_clean.columns:
    before = len(df_trf_clean)
    df_trf_clean = df_trf_clean.drop_duplicates(subset=cfg.id_col, keep="first").copy()
    print(f"Dropped duplicated {cfg.id_col}: {before - len(df_trf_clean)}")

company_text = first_non_empty_text(df_trf_clean, cfg.company_text_candidates)
extracted_company = company_text.map(extract_company).map(normalize_company)
if cfg.company_col in df_trf_clean.columns:
    existing_company = df_trf_clean[cfg.company_col].map(normalize_company)
else:
    existing_company = pd.Series(pd.NA, index=df_trf_clean.index, dtype="object")

df_trf_clean["company_name_raw"] = existing_company.fillna(extracted_company)
df_trf_clean["company_name"] = df_trf_clean["company_name_raw"].map(normalize_company)
df_trf_clean = canonicalize_company_names(df_trf_clean)
df_trf_clean["amount"] = pd.to_numeric(df_trf_clean[cfg.amount_col], errors="coerce")
df_trf_clean["dayofweek"] = df_trf_clean[cfg.date_col].dt.dayofweek
df_trf_clean["day_name"] = df_trf_clean[cfg.date_col].dt.day_name()
df_trf_clean["dayofmonth"] = df_trf_clean[cfg.date_col].dt.day
df_trf_clean = df_trf_clean.dropna(subset=[cfg.date_col, "amount"]).copy()

calendar = pd.date_range(df_trf_clean[cfg.date_col].min(), df_trf_clean[cfg.date_col].max(), freq="D", name=cfg.date_col)

print("Final cleaned TRF+ rows:", df_trf_clean.shape)
print("Date range:", df_trf_clean[cfg.date_col].min(), "->", df_trf_clean[cfg.date_col].max())
print("Known company rate:", round(float(df_trf_clean["company_found"].mean()), 3))
print("Unique known companies:", df_trf_clean.loc[df_trf_clean["company_found"], "company_name"].nunique())

display(df_trf_clean.head())

In [ ]:
manual_series_specs = [
    {
        "series_id": "EL_CORTE_INGLES_FULL",
        "company": "EL CORTE INGLES",
        "payment_bin": [0, np.inf],
        "daily_min": 0,
    },
    {
        "series_id": "MERCADONA_HIGH_CLEAN",
        "company": "MERCADONA",
        "payment_bin": [60000, np.inf],
        "daily_min": 60000,
    },
    # Add one dict per business-driven series. A series is exactly one payment interval.
]

AUTO_K = 10
PLOT_SERIES_IDS = []


def parse_payment_bin(interval) -> tuple[float, float]:
    if interval is None:
        interval = [0, np.inf]
    if len(interval) != 2:
        raise ValueError(f"payment_bin must be [min, max), got {interval!r}")
    low, high = interval
    low = -np.inf if low is None else float(low)
    high = np.inf if high is None else float(high)
    if not low < high:
        raise ValueError(f"payment_bin lower bound must be < upper bound, got {interval!r}")
    return low, high


def format_payment_bin(low: float, high: float) -> str:
    low_label = "-inf" if np.isneginf(low) else f"{low:g}"
    high_label = "inf" if np.isposinf(high) else f"{high:g}"
    return f"[{low_label}, {high_label})"


def build_manual_membership(tx: pd.DataFrame, specs: list[dict], cfg: SeriesConfig = cfg) -> pd.DataFrame:
    required = [cfg.id_col, cfg.date_col, "company_name", "amount"]
    missing = [col for col in required if col not in tx.columns]
    if missing:
        raise KeyError(f"Missing columns for manual series construction: {missing}")

    frames = []
    for spec in specs:
        series_id = str(spec["series_id"])
        company = normalize_company(spec["company"])
        if not company:
            raise ValueError(f"Could not normalize company for {series_id!r}: {spec['company']!r}")
        low, high = parse_payment_bin(spec.get("payment_bin", [0, np.inf]))
        daily_min = float(spec.get("daily_min", 0) or 0)

        selected = tx.loc[
            tx["company_name"].eq(company)
            & tx["amount"].ge(low)
            & tx["amount"].lt(high)
        ].copy()
        if selected.empty:
            continue

        selected_daily_amount = selected.groupby(cfg.date_col, observed=True)["amount"].transform("sum")
        selected = selected.loc[selected_daily_amount.ge(daily_min)].copy()
        if selected.empty:
            continue

        selected["series_id"] = series_id
        selected["company_spec"] = company
        selected["payment_bin_low"] = low
        selected["payment_bin_high"] = high
        selected["payment_bin"] = format_payment_bin(low, high)
        selected["daily_min"] = daily_min
        selected["selected_daily_amount"] = selected.groupby(cfg.date_col, observed=True)["amount"].transform("sum")
        frames.append(
            selected[
                [
                    cfg.id_col,
                    "series_id",
                    "company_spec",
                    "company_name",
                    cfg.date_col,
                    "amount",
                    "payment_bin_low",
                    "payment_bin_high",
                    "payment_bin",
                    "daily_min",
                    "selected_daily_amount",
                ]
            ]
        )

    columns = [
        cfg.id_col,
        "series_id",
        "company_spec",
        "company_name",
        cfg.date_col,
        "amount",
        "payment_bin_low",
        "payment_bin_high",
        "payment_bin",
        "daily_min",
        "selected_daily_amount",
    ]
    if not frames:
        return pd.DataFrame(columns=columns)

    membership = pd.concat(frames, ignore_index=True)
    duplicated = membership.loc[membership.duplicated(cfg.id_col, keep=False)].sort_values(cfg.id_col)
    if not duplicated.empty:
        display(duplicated.head(50))
        raise ValueError(f"Manual series specs overlap on {duplicated[cfg.id_col].nunique()} source IDs")
    return membership


def build_dense_daily_series(
    rows: pd.DataFrame,
    calendar,
    date_col: str,
    amount_col: str = "amount",
    series_col: str = "series_id",
    series_kind: str | None = None,
) -> pd.DataFrame:
    cal = pd.DatetimeIndex(calendar, name=date_col)
    columns = [date_col, series_col, amount_col]
    if series_kind is not None:
        columns.append("series_kind")
    if rows.empty:
        return pd.DataFrame(columns=columns)

    series_ids = pd.Index(sorted(rows[series_col].dropna().astype(str).unique()), name=series_col)
    sparse = rows.groupby([date_col, series_col], observed=True)[amount_col].sum()
    full_index = pd.MultiIndex.from_product([cal, series_ids], names=[date_col, series_col])
    dense = sparse.reindex(full_index, fill_value=0.0).reset_index(name=amount_col)
    if series_kind is not None:
        dense["series_kind"] = series_kind
    return dense


def series_membership_summary(membership: pd.DataFrame, cfg: SeriesConfig = cfg) -> pd.DataFrame:
    if membership.empty:
        return pd.DataFrame(
            columns=["series_id", "row_count", "active_days", "total_amount", "first_date", "last_date"]
        )
    return (
        membership.groupby("series_id", observed=True)
        .agg(
            row_count=(cfg.id_col, "nunique"),
            active_days=(cfg.date_col, "nunique"),
            total_amount=("amount", "sum"),
            first_date=(cfg.date_col, "min"),
            last_date=(cfg.date_col, "max"),
        )
        .reset_index()
        .sort_values("total_amount", ascending=False)
    )


def rows_for_series(series_id: str, membership: pd.DataFrame, tx: pd.DataFrame, cfg: SeriesConfig = cfg) -> pd.DataFrame:
    ids = membership.loc[membership["series_id"].eq(series_id), cfg.id_col]
    return tx.loc[tx[cfg.id_col].isin(ids)].copy()


def exclude_series_ids(series_ids, membership: pd.DataFrame, tx: pd.DataFrame, cfg: SeriesConfig = cfg) -> pd.DataFrame:
    ids = membership.loc[membership["series_id"].isin(series_ids), cfg.id_col]
    return tx.loc[~tx[cfg.id_col].isin(ids)].copy()


In [ ]:
manual_membership = build_manual_membership(df_trf_clean, manual_series_specs)
manual_ids = set(manual_membership[cfg.id_col])
clean_ids = set(df_trf_clean[cfg.id_col])
missing_manual_ids = manual_ids - clean_ids
assert not missing_manual_ids, f"Manual membership contains IDs absent from df_trf_clean: {list(missing_manual_ids)[:10]}"
assert not manual_membership[cfg.id_col].duplicated().any(), "Manual membership has duplicated source IDs"

manual_series_daily = build_dense_daily_series(
    manual_membership,
    calendar,
    date_col=cfg.date_col,
    amount_col="amount",
    series_col="series_id",
    series_kind="manual",
)

df_trf_after_manual = df_trf_clean.loc[~df_trf_clean[cfg.id_col].isin(manual_ids)].copy()
assert len(df_trf_clean) - len(df_trf_after_manual) == len(manual_ids)

print("Manual selected IDs:", len(manual_ids))
print("Rows left for automatic clustering:", len(df_trf_after_manual))
display(series_membership_summary(manual_membership))
display(manual_series_daily.head())


In [ ]:
def build_company_feature_frame(tx: pd.DataFrame, calendar, cfg: SeriesConfig = cfg) -> pd.DataFrame:
    cal = pd.DatetimeIndex(calendar, name=cfg.date_col)
    n_days = len(cal)
    if tx.empty:
        return pd.DataFrame()

    rows = []
    for company_key, group in tx.groupby("company_key", dropna=False, observed=True):
        company_key = "NO_COMPANY" if pd.isna(company_key) else str(company_key)
        daily = group.groupby(cfg.date_col, observed=True)["amount"].sum().reindex(cal, fill_value=0.0).astype(float)
        row_counts_by_day = group.groupby(cfg.date_col, observed=True)[cfg.id_col].nunique().reindex(cal, fill_value=0)
        active_mask = daily.ne(0)
        active_amounts = daily.loc[active_mask]
        active_days = int(active_mask.sum())
        active_dates = pd.DatetimeIndex(daily.index[active_mask.to_numpy()])
        total_amount = float(daily.sum())
        total_abs_amount = float(daily.abs().sum())
        abs_daily = daily.abs().sort_values(ascending=False)
        adi = float(n_days / active_days) if active_days else np.nan
        cv2 = safe_cv2(active_amounts)
        gaps = gap_stats(active_dates)

        if active_days:
            weekday_values = active_dates.dayofweek
            dayofmonth_values = active_dates.day
            days_in_month = active_dates.days_in_month
            month_start_share = float((active_dates.day <= 5).mean())
            month_end_share = float((active_dates.day >= days_in_month - 4).mean())
            weekend_share = float((weekday_values >= 5).mean())
            rows_per_active_day = row_counts_by_day.loc[active_mask]
            active_q50 = float(active_amounts.quantile(0.50))
            active_q90 = float(active_amounts.quantile(0.90))
            active_q95 = float(active_amounts.quantile(0.95))
            active_max = float(active_amounts.max())
            spike_threshold = active_amounts.quantile(0.95)
            spike_active_share = float(active_amounts.gt(spike_threshold).mean())
        else:
            weekday_values = []
            dayofmonth_values = []
            month_start_share = 0.0
            month_end_share = 0.0
            weekend_share = 0.0
            rows_per_active_day = pd.Series(dtype=float)
            active_q50 = active_q90 = active_q95 = active_max = 0.0
            spike_active_share = 0.0

        lead_days = pd.Series(dtype=float)
        if "TRADE_DATE" in group.columns:
            lead_days = (group[cfg.date_col] - group["TRADE_DATE"]).dt.days.dropna().astype(float)

        row = {
            "company_key": company_key,
            "company_found": bool(group["company_found"].fillna(False).any()) if "company_found" in group.columns else company_key != "NO_COMPANY",
            "row_count": int(group[cfg.id_col].nunique()),
            "active_days": active_days,
            "calendar_days": n_days,
            "total_amount": total_amount,
            "total_abs_amount": total_abs_amount,
            "mean_calendar_daily_amount": float(daily.mean()),
            "mean_active_daily_amount": float(active_amounts.mean()) if active_days else 0.0,
            "median_active_daily_amount": active_q50,
            "q90_active_daily_amount": active_q90,
            "q95_active_daily_amount": active_q95,
            "max_active_daily_amount": active_max,
            "max_day_abs_share": float(abs_daily.iloc[0] / total_abs_amount) if total_abs_amount else 0.0,
            "top3_day_abs_share": float(abs_daily.head(3).sum() / total_abs_amount) if total_abs_amount else 0.0,
            "active_rate": float(active_days / n_days) if n_days else 0.0,
            "zero_rate": float(1.0 - active_days / n_days) if n_days else 0.0,
            "adi": adi,
            "cv2": cv2,
            "longest_zero_run": longest_run(~active_mask.to_numpy()),
            "longest_active_run": longest_run(active_mask.to_numpy()),
            "weekday_entropy": normalized_entropy(weekday_values),
            "weekday_top_share": top_share(weekday_values),
            "weekend_active_share": weekend_share,
            "month_start_active_share": month_start_share,
            "month_end_active_share": month_end_share,
            "dayofmonth_entropy": normalized_entropy(dayofmonth_values),
            "dayofmonth_top_share": top_share(dayofmonth_values),
            "rows_per_active_day_mean": float(rows_per_active_day.mean()) if active_days else 0.0,
            "rows_per_active_day_max": float(rows_per_active_day.max()) if active_days else 0.0,
            "rows_per_active_day_cv2": safe_cv2(rows_per_active_day),
            "autocorr_lag_1": autocorr_at_lag(daily, 1),
            "autocorr_lag_7": autocorr_at_lag(daily, 7),
            "autocorr_lag_14": autocorr_at_lag(daily, 14),
            "autocorr_lag_28": autocorr_at_lag(daily, 28),
            "mean_abs_daily_diff": float(daily.diff().abs().dropna().mean()) if n_days > 1 else 0.0,
            "spike_active_share": spike_active_share,
            "recent_30_amount_share": share_last_days(daily, 30),
            "recent_90_amount_share": share_last_days(daily, 90),
            "recent_180_amount_share": share_last_days(daily, 180),
            "lead_days_mean": float(lead_days.mean()) if not lead_days.empty else np.nan,
            "lead_days_median": float(lead_days.median()) if not lead_days.empty else np.nan,
            "known_at_least_d1_share": float(lead_days.ge(1).mean()) if not lead_days.empty else np.nan,
            "known_same_day_share": float(lead_days.eq(0).mean()) if not lead_days.empty else np.nan,
        }
        row.update(gaps)
        row["sbc_label"] = sbc_label(row["adi"], row["cv2"])
        rows.append(row)

    return pd.DataFrame(rows).sort_values("total_abs_amount", ascending=False).reset_index(drop=True)


def prepare_cluster_matrix(features: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    numeric = features.select_dtypes(include=[np.number]).copy()
    numeric = numeric.replace([np.inf, -np.inf], np.nan)

    log_tokens = ("count", "days", "amount", "gap", "run", "diff")
    for col in numeric.columns:
        values = numeric[col].dropna()
        if values.empty:
            continue
        if values.min() >= 0 and any(token in col for token in log_tokens):
            numeric[col] = np.log1p(numeric[col])

    medians = numeric.median(numeric_only=True).fillna(0.0)
    numeric = numeric.fillna(medians).fillna(0.0)
    categorical = pd.get_dummies(features[["sbc_label"]].fillna("unknown"), prefix="sbc", dtype=float)
    matrix = pd.concat([numeric, categorical], axis=1)
    if matrix.empty:
        matrix["constant"] = 1.0

    scaler = RobustScaler()
    scaled = pd.DataFrame(
        scaler.fit_transform(matrix),
        columns=matrix.columns,
        index=features.index,
    )
    return matrix, scaled


def cluster_company_features(features: pd.DataFrame, auto_k: int) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    if features.empty:
        return features.copy(), pd.DataFrame(), pd.DataFrame()
    if auto_k < 1:
        raise ValueError("AUTO_K must be >= 1")

    _, scaled = prepare_cluster_matrix(features)
    k = min(int(auto_k), len(features))
    if k == 1:
        raw_labels = np.zeros(len(features), dtype=int)
    else:
        raw_labels = KMeans(n_clusters=k, random_state=0, n_init=50).fit_predict(scaled)

    clustered = features.copy()
    clustered["cluster_raw"] = raw_labels
    cluster_order = (
        clustered.groupby("cluster_raw", observed=True)["total_amount"]
        .sum()
        .sort_values(ascending=False)
        .index.tolist()
    )
    label_map = {cluster: f"AUTO_{i + 1:02d}" for i, cluster in enumerate(cluster_order)}
    clustered["auto_series_id"] = clustered["cluster_raw"].map(label_map)

    coords = np.zeros((len(clustered), 2), dtype=float)
    if len(clustered) >= 2 and scaled.shape[1] >= 2:
        coords = PCA(n_components=2).fit_transform(scaled)
    elif scaled.shape[1] == 1:
        coords[:, 0] = scaled.iloc[:, 0].to_numpy()
    clustered["pca_x"] = coords[:, 0]
    clustered["pca_y"] = coords[:, 1]
    clustered["plot_size"] = clustered["total_abs_amount"].clip(lower=0)

    scaled_with_cluster = scaled.assign(auto_series_id=clustered["auto_series_id"].to_numpy())
    profile = scaled_with_cluster.groupby("auto_series_id", observed=True).median().reset_index()
    return clustered, scaled, profile


In [ ]:
auto_company_features = build_company_feature_frame(df_trf_after_manual, calendar)
auto_company_clusters, auto_feature_matrix_scaled, auto_cluster_profile_scaled = cluster_company_features(
    auto_company_features,
    AUTO_K,
)

if auto_company_clusters.empty:
    auto_membership = pd.DataFrame(columns=[cfg.id_col, "series_id", "company_key", cfg.date_col, "amount"])
else:
    auto_membership = df_trf_after_manual.merge(
        auto_company_clusters[["company_key", "auto_series_id"]],
        on="company_key",
        how="inner",
        validate="many_to_one",
    ).rename(columns={"auto_series_id": "series_id"})
    auto_membership = auto_membership[[cfg.id_col, "series_id", "company_key", cfg.date_col, "amount"]]

auto_series_daily = build_dense_daily_series(
    auto_membership,
    calendar,
    date_col=cfg.date_col,
    amount_col="amount",
    series_col="series_id",
    series_kind="automatic",
)
all_series_daily = pd.concat([manual_series_daily, auto_series_daily], ignore_index=True, sort=False)

expected_removed = set(manual_membership[cfg.id_col])
actual_removed = set(df_trf_clean[cfg.id_col]) - set(df_trf_after_manual[cfg.id_col])
assert actual_removed == expected_removed, "df_trf_after_manual did not remove exactly the manual IDs"
if not all_series_daily.empty:
    expected_rows = len(calendar) * all_series_daily["series_id"].nunique()
    assert len(all_series_daily) == expected_rows, "Dense daily series should have one row per day per series_id"

print("Automatic companies:", len(auto_company_clusters))
print("Automatic rows:", len(auto_membership))
print("Total daily series:", all_series_daily["series_id"].nunique() if not all_series_daily.empty else 0)

if auto_company_clusters.empty:
    auto_cluster_summary = pd.DataFrame(
        columns=["auto_series_id", "company_count", "row_count", "active_days", "total_amount", "total_abs_amount"]
    )
else:
    auto_cluster_summary = (
        auto_company_clusters.groupby("auto_series_id", observed=True)
        .agg(
            company_count=("company_key", "nunique"),
            row_count=("row_count", "sum"),
            active_days=("active_days", "sum"),
            total_amount=("total_amount", "sum"),
            total_abs_amount=("total_abs_amount", "sum"),
        )
        .reset_index()
        .sort_values("auto_series_id")
    )

display(auto_cluster_summary)
display(auto_company_clusters.head(20))
display(all_series_daily.head())

if not auto_company_clusters.empty:
    fig = px.scatter(
        auto_company_clusters,
        x="pca_x",
        y="pca_y",
        color="auto_series_id",
        size="plot_size",
        size_max=28,
        hover_name="company_key",
        hover_data=["row_count", "active_days", "total_amount", "active_rate", "adi", "cv2", "sbc_label"],
        labels={"pca_x": "PCA 1", "pca_y": "PCA 2", "auto_series_id": "Auto series"},
        title="Company clustering map",
    )
    fig.update_layout(template="plotly_white", height=540, legend_title_text="")
    fig.show()

    fig = px.bar(
        auto_cluster_summary,
        x="auto_series_id",
        y="total_amount",
        text="company_count",
        hover_data=["row_count", "active_days", "total_abs_amount"],
        labels={"auto_series_id": "Auto series", "total_amount": "Total amount", "company_count": "Companies"},
        title="Automatic cluster total amount and company count",
    )
    fig.update_layout(template="plotly_white", height=430)
    fig.show()

    profile = auto_cluster_profile_scaled.set_index("auto_series_id")
    if not profile.empty:
        plot_cols = profile.var(axis=0).sort_values(ascending=False).head(25).index.tolist()
        fig = px.imshow(
            profile[plot_cols],
            aspect="auto",
            color_continuous_scale="RdBu_r",
            zmin=-3,
            zmax=3,
            labels={"x": "Scaled feature", "y": "Auto series", "color": "Median scaled value"},
            title="Automatic cluster profile heatmap",
        )
        fig.update_layout(template="plotly_white", height=max(360, 42 * len(profile)))
        fig.show()

if PLOT_SERIES_IDS:
    plot_series = all_series_daily.loc[all_series_daily["series_id"].isin(PLOT_SERIES_IDS)].copy()
    if not plot_series.empty:
        fig = px.line(
            plot_series,
            x=cfg.date_col,
            y="amount",
            color="series_id",
            labels={cfg.date_col: "Value date", "amount": "Daily amount", "series_id": "Series"},
            title="Selected generated daily series",
        )
        fig.update_layout(template="plotly_white", height=430, legend_title_text="")
        fig.show()
